###Anomaly Detection Model

In [32]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DB_PATH = "/content/machine_telemetry_degraded.db"

with sqlite3.connect(DB_PATH) as conn:
    dataframe_machine_degraded = pd.read_sql_query(
        "SELECT * FROM machine_telemetry",
        conn,
        parse_dates=["timestamp"]
    )

dataframe_machine_degraded.head()

healthy = dataframe_machine_degraded[dataframe_machine_degraded["machine_condition"] == "HEALTHY"].copy()

print(f"Healthy observations: {len(healthy):,}")
print(f"Total observations:   {len(dataframe_machine_degraded):,}")

features = [
    "hydraulic_pressure",
    "hydraulic_temperature",
    "vibration",
    "fuel_rate",
]

baseline = healthy[features].agg(["mean", "std"])

baseline

z_scores = pd.DataFrame(index=dataframe_machine_degraded.index)

for feature in features:
    mean = baseline.loc["mean", feature]
    std = baseline.loc["std", feature]

    z_scores[feature] = (
        (dataframe_machine_degraded[feature] - mean) / std
    )
z_scores.head()

dataframe_machine_degraded["anomaly_score"] = np.sqrt(
    (z_scores ** 2).sum(axis=1)
)

plt.figure(figsize=(16, 5))

plt.plot(
    dataframe_machine_degraded["timestamp"],
    dataframe_machine_degraded["anomaly_score"],
)

plt.axvline(
    dataframe_machine_degraded["timestamp"].min() + pd.Timedelta(hours=12),
    linestyle="--",
    label="Degradation starts",
)

plt.axvline(
    dataframe_machine_degraded["timestamp"].min() + pd.Timedelta(hours=20),
    linestyle="--",
    label="Critical phase",
)

plt.xlabel("Time")
plt.ylabel("Anomaly score")
plt.title("Baseline Anomaly Score")
plt.legend()
plt.show()

healthy_scores = dataframe_machine_degraded.loc[
    dataframe_machine_degraded["machine_condition"] == "HEALTHY",
    "anomaly_score"
]

healthy_scores.describe()

plt.figure(figsize=(12, 5))

plt.hist(
    healthy_scores,
    bins=50
)

plt.xlabel("Anomaly score")
plt.ylabel("Frequency")
plt.title("Anomaly Score Distribution - Healthy Operation")

plt.show()

dataframe_machine_degraded.groupby("machine_condition")["anomaly_score"].describe()
dataframe_machine_degraded.groupby("machine_condition")["anomaly_score"].mean()
healthy_scores.quantile(
    [0.90, 0.95, 0.975, 0.99, 0.995]
)

threshold = healthy_scores.quantile(0.99)

dataframe_machine_degraded["is_anomaly"] = (
    dataframe_machine_degraded["anomaly_score"] > threshold
)

print(f"Threshold: {threshold:.3f}")

dataframe_machine_degraded["is_anomaly"].value_counts()

plt.figure(figsize=(16, 5))

plt.plot(
    dataframe_machine_degraded["timestamp"],
    dataframe_machine_degraded["anomaly_score"],
    label="Anomaly score"
)

plt.axhline(
    threshold,
    linestyle="--",
    label=f"Threshold = {threshold:.2f}"
)

plt.axvline(
    dataframe_machine_degraded["timestamp"].min() + pd.Timedelta(hours=12),
    linestyle="--",
    label="Degradation starts"
)

plt.axvline(
    dataframe_machine_degraded["timestamp"].min() + pd.Timedelta(hours=20),
    linestyle="--",
    label="Critical phase"
)

plt.xlabel("Time")
plt.ylabel("Anomaly score")
plt.title("Anomaly Detection Threshold")

plt.legend()
plt.show()

dataframe_machine_degraded["anomaly_rate_60s"] = (
    dataframe_machine_degraded["is_anomaly"]
    .rolling(window=60)
    .mean()
)

dataframe_machine_degraded["persistent_anomaly"] = (
    dataframe_machine_degraded["anomaly_rate_60s"] >= 0.75
)

plt.figure(figsize=(16, 5))

plt.plot(
    dataframe_machine_degraded["timestamp"],
    dataframe_machine_degraded["anomaly_rate_60s"],
    label="60s anomaly rate"
)

plt.axhline(
    0.75,
    linestyle="--",
    label="Persistence threshold"
)

plt.axvline(
    dataframe_machine_degraded["timestamp"].min() + pd.Timedelta(hours=12),
    linestyle="--",
    label="Degradation starts"
)

plt.axvline(
    dataframe_machine_degraded["timestamp"].min() + pd.Timedelta(hours=20),
    linestyle="--",
    label="Critical phase"
)

plt.xlabel("Time")
plt.ylabel("Anomaly rate")
plt.title("Persistent Anomaly Detection")

plt.legend()
plt.show()

first_warning = dataframe_machine_degraded.loc[
    dataframe_machine_degraded["persistent_anomaly"],
    "timestamp"
].min()

first_warning

critical_start = (
    dataframe_machine_degraded["timestamp"].min()
    + pd.Timedelta(hours=20)
)

critical_start

lead_time = critical_start - first_warning

print(f"First persistent warning: {first_warning}")
print(f"Critical phase starts:     {critical_start}")
print(f"Lead time:                 {lead_time}")

dataframe_machine_degraded.groupby("machine_condition")["persistent_anomaly"].mean()

,persistent_anomaly
machine_condition,
CRITICAL,0.324074
DEGRADATION,0.107393
EARLY_DEGRADATION,0.000000
HEALTHY,0.000000
